In [1]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# GWO Parameters
num_wolves = 30
num_iterations = 100

# Define GWO-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown):
    num_units = len(pmin)

    # Initialize wolves (random solutions)
    wolves = np.random.uniform(pmin, pmax, (num_wolves, num_units))

    # Initialize alpha, beta, and delta wolves (best, second-best, and third-best solutions)
    alpha = None
    beta = None
    delta = None
    alpha_score = float('inf')
    beta_score = float('inf')
    delta_score = float('inf')

    for iteration in range(num_iterations):
        for i in range(num_wolves):
            # Ensure wolves respect generation limits
            wolves[i] = np.clip(wolves[i], pmin, pmax)

            # Calculate fuel cost for the wolf
            squared_P = np.multiply(wolves[i], wolves[i])
            fuel_cost = np.sum(np.add(a, np.add(np.multiply(b, wolves[i]), np.multiply(c, squared_P))))

            # Penalty for demand mismatch
            power_output = np.sum(wolves[i])
            mismatch_penalty = abs(demand - power_output) * 10000  # Large penalty for mismatch
            score = fuel_cost + mismatch_penalty

            # Update alpha, beta, and delta
            if score < alpha_score:
                delta_score, delta = beta_score, beta
                beta_score, beta = alpha_score, alpha
                alpha_score, alpha = score, wolves[i].copy()
            elif score < beta_score:
                delta_score, delta = beta_score, beta
                beta_score, beta = score, wolves[i].copy()
            elif score < delta_score:
                delta_score, delta = score, wolves[i].copy()

        # Update the positions of wolves
        for i in range(num_wolves):
            for j in range(num_units):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * r1 - 1  # Coefficient vector
                C1 = 2 * r2      # Coefficient vector
                D_alpha = abs(C1 * alpha[j] - wolves[i][j])
                X1 = alpha[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * r1 - 1
                C2 = 2 * r2
                D_beta = abs(C2 * beta[j] - wolves[i][j])
                X2 = beta[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * r1 - 1
                C3 = 2 * r2
                D_delta = abs(C3 * delta[j] - wolves[i][j])
                X3 = delta[j] - A3 * D_delta

                # Update wolf position
                wolves[i][j] = (X1 + X2 + X3) / 3

        # Normalization to meet demand
        total_power = np.sum(alpha)
        if total_power != 0:
            scaling_factor = demand / total_power
            alpha = alpha * scaling_factor

        # Ensure alpha respects generation limits
        alpha = np.clip(alpha, pmin, pmax)

        # Check for ramp rate violations and correct
        if iteration > 0:
            diff = alpha - wolves[i]
            ramp_up_mask = diff > 0
            ramp_down_mask = diff < 0
            ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
            ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
            alpha[ramp_up_mask] = wolves[i][ramp_up_mask] + ramp_up_values
            alpha[ramp_down_mask] = wolves[i][ramp_down_mask] + ramp_down_values

        # Termination condition for power balance
        if abs(demand - np.sum(alpha)) < 0.0001:
            break

    # Return the best solution
    best_power = alpha
    return best_power, iteration + 1

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP))), 1000)
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'GWO (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'GWO (ED).xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           342.379230
1      2           254.918392
2      3           293.257243
3      4           268.919829
4      5           416.026816
5      6           416.026816
6      7           448.462771
7      8           171.434832
8      9           115.398335
9     10            89.221437
10    11           145.002086
11    12            61.133375
12    13            49.081720
13    14            77.636933
14    15           146.488162
15    16           224.328185
16    17           224.328185
17    18           224.767771
18    19            10.954235
19    20            10.199526
20    21            97.426692
21    22            87.574763
22    23           111.386306
23    24            79.777393
24    25            61.870608
25    26            66.766562
26    27            79.487484
27    28            95.844468
28    29           448.777081
29    30           437.510245
30    31           